<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/14-joining.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 14 — Joining Without Losing Rows

Companion to [the chapter](https://www.ai.biz/books/python-primer/joining-data/).

The most expensive mistake in pandas, and the three lines that prevent it.


In [ ]:
import pandas as pd, numpy as np


## 1. The disaster


In [ ]:
orders = pd.DataFrame({'order_id':[1,2,3], 'customer_id':['A','B','C'],
                       'revenue':[100,200,300]})
customers = pd.DataFrame({'customer_id':['A','B','C','B'],   # B twice
                          'segment':['smb','ent','smb','ent']})

merged = orders.merge(customers, on='customer_id', how='left')
print(f'orders in : {len(orders)} rows, revenue {orders.revenue.sum()}')
print(f'after join: {len(merged)} rows, revenue {merged.revenue.sum()}  <- inflated 33%')
print(merged)


No error. No warning. Every downstream sum is now wrong.


## 2. validate= states what you believe


In [ ]:
try:
    orders.merge(customers, on='customer_id', how='left', validate='many_to_one')
except pd.errors.MergeError as e:
    print('caught immediately:', e)


## 3. The three lines worth building into a reflex


In [ ]:
clean_customers = customers.drop_duplicates('customer_id')

before = len(orders)
m = orders.merge(clean_customers, on='customer_id', how='left',
                 validate='many_to_one', indicator=True)
assert len(m) == before, f'row count changed: {before} -> {len(m)}'

print(m['_merge'].value_counts())
print(f'revenue preserved: {m.revenue.sum()}')


## 4. Why keys fail to match

### Type mismatch


In [ ]:
left  = pd.DataFrame({'k':[1,2,3], 'v':list('abc')})
right = pd.DataFrame({'k':['1','2','3'], 'w':list('xyz')})
try:
    left.merge(right, on='k')
except ValueError as e:
    print('pandas refuses:', e)
print('after casting:', len(left.merge(right.astype({'k':'int64'}), on='k')), 'matches')


### Whitespace and case


In [ ]:
a = pd.DataFrame({'city':['Delhi ','TOKYO'], 'x':[1,2]})
b = pd.DataFrame({'city':['delhi','tokyo'], 'y':[10,20]})
print('raw join      :', len(a.merge(b, on='city')), 'matches')

for d in (a, b): d['city'] = d.city.str.strip().str.lower()
print('normalised    :', len(a.merge(b, on='city')), 'matches')


### Diagnose a disappointing join


In [ ]:
lk, rk = set(a.city), set(b.city)
print(f'left only : {len(lk - rk)}  {sorted(lk - rk)[:3]}')
print(f'right only: {len(rk - lk)}  {sorted(rk - lk)[:3]}')
print(f'both      : {len(lk & rk)}')


## 5. how='left' beats how='inner' as a default


In [ ]:
o = pd.DataFrame({'id':[1,2,3,4], 'rev':[10,20,30,40], 'cust':['A','B','X','Y']})
c = pd.DataFrame({'cust':['A','B'], 'seg':['smb','ent']})

inner = o.merge(c, on='cust', how='inner')
left  = o.merge(c, on='cust', how='left', indicator=True)

print(f'inner: {len(inner)} rows, revenue {inner.rev.sum()}  <- 2 rows vanished silently')
print(f'left : {len(left)} rows, revenue {left.rev.sum()}  <- failures visible')
print(left['_merge'].value_counts())


## 6. A safe_merge helper worth keeping


In [ ]:
def safe_merge(left, right, on, how='left', validate='many_to_one', **kw):
    before = len(left)
    out = left.merge(right, on=on, how=how, validate=validate,
                     indicator=True, **kw)
    if how == 'left' and len(out) != before:
        raise ValueError(f'row count changed: {before} -> {len(out)}')
    n_unmatched = (out['_merge'] == 'left_only').sum()
    if n_unmatched:
        print(f'warning: {n_unmatched} rows found no match')
    return out.drop(columns='_merge')

print(safe_merge(o, c, on='cust'))


## 7. merge_asof for point-in-time joins

Attaching today's attributes to a past event is leakage. This attaches what was true at the time.


In [ ]:
events = pd.DataFrame({
    'ts': pd.to_datetime(['2026-01-15','2026-03-15','2026-06-15']),
    'customer': ['A','A','A'], 'amount': [10,20,30]}).sort_values('ts')
segments = pd.DataFrame({
    'valid_from': pd.to_datetime(['2026-01-01','2026-04-01']),
    'customer': ['A','A'], 'segment': ['smb','enterprise']}).sort_values('valid_from')

print(pd.merge_asof(events, segments, left_on='ts', right_on='valid_from',
                    by='customer', direction='backward'))
print()
print('The June event gets enterprise; the earlier ones get smb. Correct.')


## Try it yourself

1. Change `validate` to `'one_to_one'` on the clean merge and see it fail.
2. Add a `NaN` customer_id and confirm it never matches.
3. Set `direction='forward'` on the as-of join and explain why that is leakage.
